# MicroDuck JumpSpin 360 — Colab A100 Trainer

이 노트북은 실험적 **JumpSpin 360** task를 학습합니다.

- Task: `Mjlab-JumpSpin-Flat-MicroDuck`
- 4096 parallel environments
- 180° → 270° → 360° curriculum
- standing / mid-air / landing reverse curriculum
- 목표 10,000 PPO iterations
- task 자체 checkpoint 간격 100 iterations
- Google Drive 자동 backup + resume

실제 로봇 배포 전에는 simulation rollout과 착지 충격/관절 한계를 반드시 검증하세요.

먼저 **Runtime → Change runtime type → A100 GPU**를 선택하세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], check=False)

## 1. 설정

In [ ]:
TASK_ID = "Mjlab-JumpSpin-Flat-MicroDuck"
EXPERIMENT_NAME = "jump_spin"
NUM_ENVS = 4096
TARGET_ITERS = 10000
SYNC_INTERVAL = 10
DRIVE_ROOT = "/content/drive/MyDrive/microduck-training"
SMOKE_TEST = True
VIDEO_TRIALS = 5
VIDEO_WIDTH = 720
VIDEO_HEIGHT = 720


## 2. 저장소 clone + 설치

In [ ]:
import os, shutil, subprocess

REPO_DIR = "/content/microduck"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run([
        "git", "clone",
        "https://github.com/namyikim/microduck.git",
        REPO_DIR,
    ], check=True)

subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/setup_colab.sh"],
    check=True,
)

## 3. 학습 시작 / 자동 resume

In [ ]:
import os, subprocess

env = os.environ.copy()
env.update({
    "TASK_ID": TASK_ID,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "NUM_ENVS": str(NUM_ENVS),
    "TARGET_ITERS": str(TARGET_ITERS),
    "SYNC_INTERVAL": str(SYNC_INTERVAL),
    "DRIVE_ROOT": DRIVE_ROOT,
    "SMOKE_TEST": "1" if SMOKE_TEST else "0",
    "WANDB_MODE": env.get("WANDB_MODE", "offline"),
    "REPO_DIR": REPO_DIR,
    "PYTHONUNBUFFERED": "1",
})

print("=== MicroDuck training starting ===", flush=True)
print("Output should appear continuously below this cell.", flush=True)
print(f"Backup log: {DRIVE_ROOT}/training.log", flush=True)

process = subprocess.Popen(
    ["bash", f"{REPO_DIR}/scripts/train_colab.sh"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)


# 학습이 정상 종료되면 최신 checkpoint로 자동 rollout 영상을 생성합니다.
from pathlib import Path
import re

print("\n=== JumpSpin video rendering ===", flush=True)

local_root = Path(REPO_DIR) / "logs" / "rsl_rl" / EXPERIMENT_NAME
checkpoints = []
for p in local_root.rglob("model_*.pt"):
    m = re.fullmatch(r"model_(\d+)\.pt", p.name)
    if m:
        checkpoints.append((int(m.group(1)), p))

if not checkpoints:
    raise RuntimeError(f"No JumpSpin checkpoint found under {local_root}")

checkpoints.sort(key=lambda x: x[0])
latest_iter, latest_checkpoint = checkpoints[-1]
video_dir = Path(DRIVE_ROOT) / "videos"
video_dir.mkdir(parents=True, exist_ok=True)

print(f"Latest checkpoint: {latest_checkpoint}", flush=True)
print(f"Creating {VIDEO_TRIALS} rollout videos in {video_dir}", flush=True)

render_cmd = [
    "uv", "run", "python", "scripts/render_jump_spin.py",
    "--checkpoint", str(latest_checkpoint),
    "--output-dir", str(video_dir),
    "--trials", str(VIDEO_TRIALS),
    "--width", str(VIDEO_WIDTH),
    "--height", str(VIDEO_HEIGHT),
]

render = subprocess.Popen(
    render_cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert render.stdout is not None
for line in render.stdout:
    print(line, end="", flush=True)

render_code = render.wait()
if render_code != 0:
    raise subprocess.CalledProcessError(render_code, render.args)

print("\n=== Generated JumpSpin videos ===", flush=True)
from IPython.display import Video, display

videos = sorted(video_dir.glob("jump_spin_trial_*.mp4"))
for video in videos:
    print(video)
    display(Video(str(video), embed=True, width=VIDEO_WIDTH))


## 4. 자동 평가 영상

학습이 목표 iteration까지 끝나면 같은 셀에서 최신 checkpoint를 자동으로 선택합니다.

1. play-mode의 정상 standing 상태에서 시작
2. 5회 독립 rollout
3. 각 episode 전체를 MP4로 기록
4. Google Drive의 `videos/`에 저장
5. `evaluation_summary.json`에 checkpoint, reward, 관측된 회전 진행량 기록
6. Colab 화면에서 생성된 영상을 바로 재생

영상 생성은 학습이 정상 종료된 뒤 실행됩니다. 세션이 중간에 끊기면 checkpoint에서 학습을 재개한 후 목표에 도달했을 때 생성됩니다.


## 5. Drive checkpoint 확인


In [ ]:
from pathlib import Path
import re

root = Path(DRIVE_ROOT) / "logs" / "rsl_rl" / EXPERIMENT_NAME
items = []
if root.exists():
    for p in root.rglob("model_*.pt"):
        m = re.fullmatch(r"model_(\d+)\.pt", p.name)
        if m:
            items.append((int(m.group(1)), p))
items.sort(key=lambda x: x[0])

if not items:
    print("아직 checkpoint가 없습니다.")
else:
    for n, p in items[-10:]:
        print(f"{n + 1:6d} iterations (approx.)  {p}")
    print("Latest:", items[-1][1])